# Abu Hureirah chip trainer (blueprint classes)

Fits a multinomial logistic model on the **same 7 features** the sitroom uses (`blobs, hv, edge, exg, red, delta, meanL`).

Classes: `none camp veh berm burn wreck_air wreck_bldg cargo`.

This is morphology, not weapons ID. A burn is a burn. A pad is a pad. Occupancy stays a human call.

**Steps:** Runtime → GPU. Upload `ahsr-chip-samples.json` (Export labels from the sitroom). Run all. Download `ahsr-chip-weights.json`. Import in the sitroom.

In [ ]:
!pip -q install scikit-learn numpy
from google.colab import files
import json, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
print('upload ahsr-chip-samples.json')
uploaded = files.upload()
raw = json.loads(next(iter(uploaded.values())))
samples = raw.get('samples', raw if isinstance(raw, list) else [])
print('n=', len(samples))

In [ ]:
FEATS = ['blobs','hv','edge','exg','red','delta','meanL']
KLASS = ['none','camp','veh','berm','burn','wreck_air','wreck_bldg','cargo']
X, y = [], []
for s in samples:
    f = s.get('features') or {}
    k = s.get('klass') or 'none'
    if k not in KLASS: continue
    # label 0 => treat as 'none' (rejected morphology)
    if s.get('label') == 0: k = 'none'
    X.append([float(f.get(n, 0) or 0) for n in FEATS])
    y.append(k)
X, y = np.array(X, float), np.array(y)
print({k: int((y==k).sum()) for k in KLASS})
if len(X) < 8:
    raise SystemExit('Need more Confirm/Reject samples. Label 20+ chips in the sitroom first.')

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=400, multi_class='multinomial', class_weight='balanced')),
])
pipe.fit(X, y)
print('train acc', round(pipe.score(X, y), 3))
clf, scaler = pipe.named_steps['clf'], pipe.named_steps['scaler']
# Map sklearn coef_ back to sitroom weights: w' = coef / scale; bias' = intercept - coef·mean/scale
scale = scaler.scale_
mean = scaler.mean_
classes = list(clf.classes_)
weights = {}
for k in KLASS:
    if k not in classes:
        weights[k] = {n: 0.0 for n in FEATS}
        weights[k]['bias'] = -1.5 if k != 'none' else 0.2
        continue
    i = classes.index(k)
    coef = clf.coef_[i] if clf.coef_.ndim > 1 else clf.coef_[0]
    intercept = float(clf.intercept_[i] if len(np.atleast_1d(clf.intercept_)) > 1 else clf.intercept_[0])
    w = coef / scale
    b = intercept - float(np.dot(coef, mean / scale))
    weights[k] = {n: float(w[j]) for j, n in enumerate(FEATS)}
    weights[k]['bias'] = float(b)
out = {'version': 1, 'kind': 'ahsr-chip-weights', 'weights': weights}
open('ahsr-chip-weights.json','w').write(json.dumps(out, indent=2))
files.download('ahsr-chip-weights.json')

## Optional Path B — Sentinel HLS chips

If you also have lat/lon on each sample, you can fetch NASA GIBS HLS tiles and train a tiny CNN. The sitroom **cannot load ONNX yet** — stick to Path A (`ahsr-chip-weights.json`) for integration. Path B is for your own offline scoring.